# Meteorite Baseline Training

这个 notebook 是当前项目的主训练入口。

设计原则：
- notebook 负责改参数、启动训练、看结果
- `models/` 和 `tools/` 负责真正的底层逻辑
- 这样既适合学习，也适合后续在服务器上长期维护

## 1. 导入依赖

In [ ]:
from pathlib import Path
import sys

# 动态获取当前 Notebook 所在的目录作为项目根目录
PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("项目根目录:", PROJECT_ROOT)
print("models 文件夹是否存在:", (PROJECT_ROOT / "models").exists())
print("tools 文件夹是否存在:", (PROJECT_ROOT / "tools").exists())

In [ ]:
from pathlib import Path
import json
import pandas as pd


import torch
import torch.nn as nn
from torch.utils.data import DataLoader

#near_deplicate审查
import numpy as np
from PIL import Image, ImageOps
from tqdm.auto import tqdm


from models import build_model
from tools.dataset_tools import (
    MeteoriteDataset,
    attach_md5_group_ids,
    group_aware_train_val_split,
    load_test_dataframe,
    load_train_dataframe,
    stratified_sample_dataframe,
    stratified_train_val_split,
)


from tools.submission_tools import make_submission, predict_on_test
from tools.train_tools import (
    IntegerLabelBCEWithLogitsLoss,
    build_bce_pos_weight,
    evaluate,
    set_seed,
    train_one_epoch,
)
from tools.transforms_tools import build_eval_transforms, build_train_transforms
from tools.visualize_tools import plot_label_distribution, plot_training_curves, save_sample_grid


## 2. 配置训练参数

这里是你最常改的地方。

这版会自动把重要参数编码进实验目录名，并把完整参数保存到 `run_summary.json`。

`loss_name` 可选 `cross_entropy` 或 `bce_with_logits`。

In [ ]:
from datetime import datetime

CONFIG = {
    'data_root': PROJECT_ROOT,
    'output_base_dir': PROJECT_ROOT / 'runs',

    'model_name': 'dinov2_small',
    'pretrained': True,

    'epochs': 10,
    'batch_size': 16,
    'img_size': 224,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'val_ratio': 0.2,
    'scheduler': 'cosine',
    'loss_name': 'cross_entropy',  # 可选: 'cross_entropy', 'bce_with_logits'

    'subset_size': 0,
    'test_subset_size': 0,

    'seed': 42,
    'num_workers': 4,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}



def format_float_for_name(value):
    text = f"{value:g}"
    return text.replace("+", "")


base_exp_name = '_'.join([
    f"img{CONFIG['img_size']}",
    CONFIG['model_name'],
    'adamw',
    f"lr{format_float_for_name(CONFIG['lr'])}",
    f"wd{format_float_for_name(CONFIG['weight_decay'])}",
    f"bs{CONFIG['batch_size']}",
    f"val{format_float_for_name(CONFIG['val_ratio'])}",
    f"seed{CONFIG['seed']}",
    f"sch{CONFIG['scheduler']}",
    f"loss{CONFIG['loss_name']}",
    f"pre{int(bool(CONFIG['pretrained']))}",
    f"subset{CONFIG['subset_size'] if CONFIG['subset_size'] and CONFIG['subset_size'] > 0 else 'full'}",
    f"e{CONFIG['epochs']}",
])

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG['experiment_name'] = f"{timestamp}_{base_exp_name}"
CONFIG['output_dir'] = CONFIG['output_base_dir'] / CONFIG['experiment_name']
CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}, ensure_ascii=False, indent=2))


## 3. 固定随机种子并读取数据

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

dataset_dir = PROJECT_ROOT / "Dataset"

# 查看所有子目录
for sub in sorted(dataset_dir.iterdir()):
    if sub.is_dir():
        img_count = len(list(sub.glob("*"))) + len(list(sub.glob("*/*")))  # 粗略统计
        print(f"{sub.name}/")
        for subsub in sorted(sub.iterdir()):
            if subsub.is_dir():
                print(f"  {subsub.name}/ ({len(list(subsub.glob('*')))} 张图片)")


In [ ]:
def count_cross_split_groups(df_train, df_val, group_col):
    """
    统计某个分组列在 train / val 同时出现的组数量。
    理论上如果按这个 group_col 做 group-aware split，结果应该是 0。
    """
    if group_col is None:
        return None
    if group_col not in df_train.columns or group_col not in df_val.columns:
        return None

    train_groups = set(df_train[group_col].dropna().astype(str).tolist())
    val_groups = set(df_val[group_col].dropna().astype(str).tolist())
    return len(train_groups.intersection(val_groups))


def drop_exact_duplicates_in_train(df, group_col='group_id'):
    """
    训练集内部 exact duplicate 去重：
    同一个 md5 group 只保留 1 张。
    """
    if group_col not in df.columns:
        raise ValueError(f'列 {group_col} 不存在，无法去重。')

    candidate_path_cols = ['img_path', 'image_path', 'filepath', 'file_path', 'path']
    sort_cols = [col for col in candidate_path_cols if col in df.columns]

    if sort_cols:
        dedup_df = (
            df.sort_values([group_col] + sort_cols)
              .drop_duplicates(subset=group_col, keep='first')
              .reset_index(drop=True)
        )
    else:
        dedup_df = (
            df.sort_values(group_col)
              .drop_duplicates(subset=group_col, keep='first')
              .reset_index(drop=True)
        )

    removed_count = len(df) - len(dedup_df)
    return dedup_df, removed_count


def compute_dhash_int(image_path, hash_size=8):
    """
    计算单张图片的 dHash，返回一个整数。
    """
    image = Image.open(image_path)
    image = ImageOps.exif_transpose(image).convert('L')
    image = image.resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)

    pixels = np.asarray(image, dtype=np.int16)
    diff = pixels[:, 1:] > pixels[:, :-1]

    hash_value = 0
    for bit in diff.flatten():
        hash_value = (hash_value << 1) | int(bit)

    return hash_value


def hamming_distance_int(x, y):
    """
    计算两个整数哈希的汉明距离。
    pandas 在含缺失值的列里可能把整数哈希提升成 float，
    这里显式转成 Python int，避免位运算类型报错。
    """
    return (int(x) ^ int(y)).bit_count()


class UnionFind:
    """
    并查集，用来把彼此相似的 md5 group 合并成一个 near-duplicate family。
    """
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1


def build_near_duplicate_families_from_md5_groups(
    df,
    path_col='img_path',
    md5_group_col='group_id',
    threshold=6,
    hash_size=8,
):
    """
    先按 md5 group 选代表图，再用 dHash 把高相似 group 合并成 near-duplicate family。
    """
    if md5_group_col not in df.columns:
        raise ValueError(f'列 {md5_group_col} 不存在，无法构建 near-duplicate family。')
    if path_col not in df.columns:
        raise ValueError(f'列 {path_col} 不存在，无法读取图片。')

    rep_df = (
        df.sort_values([md5_group_col, path_col])
          .drop_duplicates(subset=md5_group_col, keep='first')
          .reset_index(drop=True)
          .copy()
    )

    dhash_list = []
    failed_paths = []

    for image_path in tqdm(rep_df[path_col].tolist(), desc='计算 md5-group 代表图 dHash'):
        try:
            dhash_list.append(compute_dhash_int(image_path, hash_size=hash_size))
        except Exception:
            dhash_list.append(None)
            failed_paths.append(str(image_path))

    # dHash 可能达到 64 位无符号整数上限，不能用 pandas 的有符号 Int64 保存。
    # 这里保持为 object 列，元素是 Python int / None，避免溢出。
    rep_df['dhash'] = pd.Series(dhash_list, dtype='object')

    valid_mask = rep_df['dhash'].notna()
    valid_rep_df = rep_df[valid_mask].reset_index(drop=True).copy()
    valid_rep_df['dhash'] = valid_rep_df['dhash'].map(int)

    n = len(valid_rep_df)
    uf = UnionFind(n)

    for i in tqdm(range(n), desc='构建 near-duplicate family'):
        hi = valid_rep_df.loc[i, 'dhash']
        for j in range(i + 1, n):
            hj = valid_rep_df.loc[j, 'dhash']
            if hamming_distance_int(hi, hj) <= threshold:
                uf.union(i, j)

    root_to_family = {}
    family_ids = []

    for i in range(n):
        root = uf.find(i)
        if root not in root_to_family:
            root_to_family[root] = f'nearfam_{len(root_to_family):05d}'
        family_ids.append(root_to_family[root])

    valid_rep_df['near_dup_family_id'] = family_ids

    invalid_rep_df = rep_df[~valid_mask].copy()
    invalid_rep_df['near_dup_family_id'] = [
        f'nearfam_failed_{i:05d}' for i in range(len(invalid_rep_df))
    ]

    rep_family_df = pd.concat(
        [
            valid_rep_df[[md5_group_col, 'dhash', 'near_dup_family_id']],
            invalid_rep_df[[md5_group_col, 'dhash', 'near_dup_family_id']],
        ],
        axis=0,
        ignore_index=True,
    )


    out_df = df.merge(
        rep_family_df[[md5_group_col, 'near_dup_family_id', 'dhash']],
        on=md5_group_col,
        how='left',
    )

    return out_df, rep_family_df, failed_paths


In [ ]:
set_seed(CONFIG['seed'])
device = torch.device(CONFIG['device'])
print('device =', device)

# 1. 仅仅读取老师提供的官方 train.csv 数据
full_train_df = load_train_dataframe(CONFIG['data_root'])
print('完整训练样本数 =', len(full_train_df))
print(f"当前训练样本类别分布:\n{full_train_df['label'].value_counts().sort_index()}")

# 2. 采样（如果 subset_size = 0 则使用全部数据）
train_pool_df = stratified_sample_dataframe(
    df=full_train_df,
    label_col='label',
    sample_size=CONFIG['subset_size'],
    seed=CONFIG['seed'],
)
print('抽样后训练池样本数 =', len(train_pool_df))

# 3. 先做 md5 exact duplicate 分组
train_pool_df = attach_md5_group_ids(train_pool_df)
md5_group_col = 'group_id'
path_col = 'img_path'

print('唯一 md5 group 数 =', train_pool_df[md5_group_col].nunique())

# 4. 在 md5 group 基础上构建 near-duplicate family
near_dup_threshold = 6
near_dup_hash_method = 'dhash'

train_pool_df, near_dup_rep_df, near_dup_failed_paths = build_near_duplicate_families_from_md5_groups(
    df=train_pool_df,
    path_col=path_col,
    md5_group_col=md5_group_col,
    threshold=near_dup_threshold,
    hash_size=8,
)

print('near-duplicate family 数 =', train_pool_df['near_dup_family_id'].nunique())
print('dHash 失败图片数 =', len(near_dup_failed_paths))

# 5. 按 near-duplicate family 做 group-aware split (划分验证集)
split_method = 'group_aware_neardup_dhash'
group_key = 'near_dup_family_id'

train_df, val_df = group_aware_train_val_split(
    df=train_pool_df,
    label_col='label',
    group_col=group_key,
    val_ratio=CONFIG['val_ratio'],
    seed=CONFIG['seed'],
)
print('切分后: train =', len(train_df), 'val =', len(val_df))

# 6. 做切分泄漏审计
cross_split_dup_count = count_cross_split_groups(train_df, val_df, md5_group_col)
cross_split_neardup_count = count_cross_split_groups(train_df, val_df, group_key)

print('split_method =', split_method)
print('group_key =', group_key)
print('cross_split_dup_count =', cross_split_dup_count)
print('cross_split_neardup_count =', cross_split_neardup_count)

# 7. 训练集内部 exact duplicate 去重
train_size_before_exact_dedup = len(train_df)
train_df, train_exact_dup_removed = drop_exact_duplicates_in_train(
    train_df,
    group_col=md5_group_col,
)
train_size_after_exact_dedup = len(train_df)

print('训练集 exact duplicate 去重前 =', train_size_before_exact_dedup)
print('训练集 exact duplicate 去重后 =', train_size_after_exact_dedup)
print('训练集移除的重复样本数 =', train_exact_dup_removed)

if cross_split_dup_count not in (None, 0):
    print(f'[WARN] md5 重复组仍跨 train/val: {cross_split_dup_count}')

if cross_split_neardup_count not in (None, 0):
    print(f'[WARN] near-duplicate family 仍跨 train/val: {cross_split_neardup_count}')

## 4. 保存切分信息和基础可视化

In [ ]:
split_dir = CONFIG['output_dir'] / 'splits'
fig_dir = CONFIG['output_dir'] / 'figures'
split_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

train_df[['id', 'label', 'img_path']].to_csv(split_dir / 'train_split.csv', index=False)
val_df[['id', 'label', 'img_path']].to_csv(split_dir / 'val_split.csv', index=False)

plot_label_distribution(train_df, fig_dir / 'train_label_distribution.png')
plot_label_distribution(val_df, fig_dir / 'val_label_distribution.png')
print('已保存类别分布图')


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from pathlib import Path
from sklearn.svm import SVC
# 导入读取测试集的工具
from tools.dataset_tools import load_test_dataframe

# ==========================================
# 0. 读取测试集数据
# ==========================================
print("📂 读取测试集数据...")
test_df = load_test_dataframe(CONFIG['data_root'])

# ==========================================
# 1. 配置 TTA (测试时增强) 预处理管线
# ==========================================
print("🚀 初始化 DINOv2 预处理 (含 TTA 增强)...")

# 这是最基础的预处理（用于训练集和 TTA 的基础视角）
base_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# 我们为测试集定义 3 种不同的“看图角度”
tta_transforms = [
    base_transform, # 视角 1: 原图
    transforms.Compose([transforms.RandomHorizontalFlip(p=1.0), base_transform]), # 视角 2: 水平翻转
    transforms.Compose([transforms.RandomVerticalFlip(p=1.0), base_transform])    # 视角 3: 垂直翻转
]

# 训练集照旧，只用原图
train_dataset = MeteoriteDataset(train_df, transforms=base_transform)
extract_train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False, num_workers=CONFIG['num_workers'])

# ==========================================
# 2. 从你刚克隆的本地目录加载模型（零网络请求）
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚙️ 正在从本地目录 /data/my-dinov2-repo 加载模型...")

# ✅ 直接使用你用 Git 克隆下来的本地路径
local_repo_dir = '/data/my-dinov2-repo'

dinov2_model = torch.hub.load(
    repo_or_dir=local_repo_dir,
    model='dinov2_vitl14_reg',
    source='local'
)
dinov2_model.eval()
dinov2_model.to(device)

# ==========================================
# 3. 提取特征 (修改测试集的提取逻辑)
# ==========================================
# ... (前面的 extract_embeddings 函数定义保持不变) ...

print("⏳ 正在提取训练集特征 (单视角)...")
train_X, train_y = extract_embeddings(extract_train_loader, dinov2_model, device, is_test=False)

print("⏳ 正在提取测试集特征 (TTA 多视角融合)...")
test_X_tta_list = []

# 循环 3 次，每次用不同的变换方法提取一遍测试集
for i, t in enumerate(tta_transforms):
    print(f"   -> 正在提取 TTA 视角 {i+1}/3 ...")
    # 临时给测试集换上当前的变换方法
    test_dataset = MeteoriteDataset(test_df, transforms=t)
    extract_test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=CONFIG['num_workers'])
    
    # 提取当前视角的特征
    curr_test_X, test_ids = extract_embeddings(extract_test_loader, dinov2_model, device, is_test=True)
    test_X_tta_list.append(curr_test_X)

# ⭐️ 终极魔法：将 3 个视角的特征在数学空间中按元素求平均
test_X = np.mean(test_X_tta_list, axis=0)

# 对平均后的特征再次进行 L2 归一化 (保证送入 SVM/kNN 的向量长度依然是 1)
from sklearn.preprocessing import normalize
test_X = normalize(test_X, norm='l2', axis=1)

print("✅ TTA 特征融合完毕！")

# ==========================================
# 4. 轻量级模型训练 (5折交叉验证)
# ==========================================
print("🧠 开始训练 LR 和 kNN 分类器...")
n_splits = 5
seed = 42
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

lr_test_preds = np.zeros(len(test_X))
knn_test_preds = np.zeros(len(test_X))

for fold, (train_idx, val_idx) in enumerate(skf.split(train_X, train_y)):
    print(f"--- Fold {fold + 1}/{n_splits} ---")
    X_tr, y_tr = train_X[train_idx], train_y[train_idx]
    
    # Logistic Regression
    lr_model = make_pipeline(
        StandardScaler(),
        SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True, random_state=seed + fold)
    )
    lr_model.fit(X_tr, y_tr)
    # 因为上面开启了 probability=True，这里依然可以使用 predict_proba
    lr_test_preds += lr_model.predict_proba(test_X)[:, 1] / n_splits
    
    # kNN
    knn_model = KNeighborsClassifier(n_neighbors=25, weights='distance', metric='cosine')
    knn_model.fit(X_tr, y_tr)
    knn_test_preds += knn_model.predict_proba(test_X)[:, 1] / n_splits

# ==========================================
# 5. Rank Average 与 Top-85 融合提交
# ==========================================
print("🏆 生成最终融合结果...")
# 对这两个 score 做简单平均
final_scores = (lr_test_preds + knn_test_preds) / 2.0

# 清洗出纯粹的文件名
clean_test_ids = [Path(p).name for p in test_ids]

sub_df = pd.DataFrame({'id': clean_test_ids, 'prob': final_scores})
sub_df = sub_df.sort_values(by='prob', ascending=False).reset_index(drop=True)

# 强制将排名前 85 的设为正类 (陨石)
top_k = 85
sub_df['label'] = 0
sub_df.loc[:top_k - 1, 'label'] = 1

# 按照官方样本顺序对齐并保存
template_df = pd.read_csv(CONFIG['data_root'] / 'sample_submission.csv')
template_df['label'] = template_df['id'].map(sub_df.set_index('id')['label'])
template_df['label'] = template_df['label'].astype(int)

output_csv = CONFIG['output_dir'] / 'submission_dinov2_L14_rankavg_top85.csv'
template_df.to_csv(output_csv, index=False)

print(f"✅ 任务完成！提分版提交文件已保存至: {output_csv}")